In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Install Required Libraries
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q -U transformers accelerate bitsandbytes
!pip install -q -U langchain langchain-community langchain-core langchain-text-splitters
!pip install -q -U sentence-transformers faiss-gpu pypdf pyvis pydantic pymupdf easyocr
!pip install -q -U pyngrok matplotlib gradio
!pip install easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 94.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 3.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━

In [4]:
# Standard Imports & Environment Verification
import os
import re
import json
import ast
import html
import fitz  
import torch
import matplotlib.pyplot as plt
from typing import List, Dict, Any, Tuple

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Authentication & Hardware Check
print(f"⚡ PyTorch Version: {torch.__version__}")
print(f"⚡ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"⚡ GPU Device: {torch.cuda.get_device_name(0)}")

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ HuggingFace Login Successful!")

⚡ PyTorch Version: 2.10.0+cu128
⚡ CUDA Available: True
⚡ GPU Device: Tesla T4
✅ HuggingFace Login Successful!


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_community.llms import HuggingFacePipeline

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

# Configure 4-bit Quantization (NF4) for Low VRAM Usage
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

print(f"⌛ Loading {MODEL_ID} with 4-bit quantization on GPU...")

/tmp/ipykernel_58/2030992329.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


⌛ Loading mistralai/Mistral-7B-Instruct-v0.2 with 4-bit quantization on GPU...


In [6]:
# Initialize Tokenizer & Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [7]:
# Create Generation Pipeline & LangChain Wrapper
text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=700,
    temperature=0.3,
    top_p=0.9,
    repetition_penalty=1.15,
    do_sample=True,
    return_full_text=False
)

llm = HuggingFacePipeline(pipeline=text_pipeline)
print(f"✅ Quantized Mistral Loaded! VRAM allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'do_sample', 'top_p', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Quantized Mistral Loaded! VRAM allocated: 1.61 GB


/tmp/ipykernel_58/3316201824.py:14: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_pipeline)


In [8]:
# ------------------------------------------
# Safe LLM Invocation Helper
# ------------------------------------------
def invoke_llm(prompt: str) -> str:
    """Executes the LLM pipeline safely and returns cleaned string output."""
    try:
        response = llm.invoke(prompt)
        if isinstance(response, str):
            return response.strip()
        return str(response).strip()
    except Exception as e:
        print(f"⚠️ LLM Invocation Error: {str(e)}")
        return ""

In [9]:
# ==========================================
# INGESTION & RAG INDEXING PIPELINE
# ==========================================
import os
import easyocr
import numpy as np
import fitz  
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("⏳ Initializing Multilingual Embedding Engine...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cuda"}
)

print("⏳ Initializing EasyOCR Engine (Arabic & English)...")
ocr_reader = easyocr.Reader(['ar', 'en'], gpu=True)

# Global Storage States
vector_store = None
full_book_chunks = []
full_book_text = ""
character_registry = {}
book_relationships = []

CARD_PALETTE = ["#818CF8", "#38BDF8", "#34D399", "#F472B6", "#FBBF24", "#A78BFA"]

# ------------------------------------------
# PDF Processing & FAISS Indexing (With OCR Fallback)
# ------------------------------------------
def process_pdf(pdf_file) -> str:
    global vector_store, full_book_chunks, full_book_text, character_registry, book_relationships
    
    if pdf_file is None:
        return "⚠️ Please select and upload a valid PDF file first."

    if isinstance(pdf_file, str):
        file_path = pdf_file
    elif hasattr(pdf_file, "name"):
        file_path = pdf_file.name
    elif isinstance(pdf_file, dict) and "name" in pdf_file:
        file_path = pdf_file["name"]
    else:
        return "❌ Error: Could not extract file path from uploaded object."

    try:
        character_registry = {}
        book_relationships = []
        
        doc = fitz.open(file_path)
        pages_text = [page.get_text("text") for page in doc]
        total_pages = len(pages_text)
        full_book_text = "\n".join(pages_text).strip()

        # Fall back to EasyOCR if standard text extraction yields little to no text
        if len(full_book_text) < 100:
            print("⚠️ Standard extraction returned little/no text. Running Arabic/English OCR...")
            ocr_text_list = []
            
            for page_num in range(total_pages):
                page = doc[page_num]
                pix = page.get_pixmap(dpi=150)
                img = np.frombuffer(pix.samples, dtype=np.uint8).reshape((pix.height, pix.width, 3))
                
                results = ocr_reader.readtext(img, detail=0)
                page_str = " ".join(results)
                ocr_text_list.append(page_str)
                
            full_book_text = "\n".join(ocr_text_list).strip()

        if not full_book_text:
            return "❌ Error: Could not extract text from PDF (even with OCR)."

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=150,
            separators=["\n\n", "\n", "،", ".", " ", ""]
        )
        
        full_book_chunks = text_splitter.split_text(full_book_text)
        vector_store = FAISS.from_texts(full_book_chunks, embedding_model)
        
        file_name = os.path.basename(file_path)
        return (f"✅ Successfully processed '{file_name}'!\n"
                f"• Total Pages Extracted: {total_pages}\n"
                f"• Total Words: {len(full_book_text.split()):,}\n"
                f"• Chunks Indexed: {len(full_book_chunks)}\n"
                f"• Ready for Q&A and Character Analysis!")
    except Exception as e:
        return f"❌ Failed to parse PDF: {str(e)}"

# ------------------------------------------
# RAG Search QA
# ------------------------------------------
def answer_rag_question(question: str) -> str:
    global vector_store
    
    if not question or not question.strip():
        return "⚠️ Please type a question."
        
    if vector_store is None:
        return "⚠️ Please upload and index a PDF book first!"
    
    try:
        docs = vector_store.similarity_search(question, k=4)
        context = "\n---\n".join([d.page_content for d in docs])
        
        prompt = f"""[INST] Answer the question using ONLY the provided book context excerpts.

Context:
{context}

Question: {question} [/INST]"""

        return invoke_llm(prompt)
    except Exception as e:
        return f"❌ Error during search: {str(e)}"

⏳ Initializing Multilingual Embedding Engine...


/tmp/ipykernel_58/3205068483.py:13: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Initializing EasyOCR Engine (Arabic & English)...


Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [10]:
# ==========================================
# CHARACTER ANALYSIS & ENTITY EXTRACTION
# ==========================================
import re
import ast
import json
from typing import List, Dict, Any

# ------------------------------------------
# Safe LLM List & JSON Normalizer
# ------------------------------------------
def parse_llm_list(raw_response: str) -> List[Dict[str, Any]]:
    """Extracts and normalizes list structures from messy or wrapped LLM outputs."""
    if not raw_response or not raw_response.strip():
        return []

    cleaned = raw_response.strip()
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned, flags=re.MULTILINE)
    cleaned = re.sub(r'\s*```$', '', cleaned, flags=re.MULTILINE)

    data = None
    try:
        data = json.loads(cleaned)
    except Exception:
        array_match = re.search(r'\[\s*\{.*\}\s*\]', cleaned, re.DOTALL)
        if array_match:
            try:
                data = json.loads(array_match.group(0))
            except Exception:
                try:
                    data = ast.literal_eval(array_match.group(0))
                except Exception:
                    pass

    if isinstance(data, dict):
        for key in ["characters", "data", "items", "results", "people"]:
            if key in data and isinstance(data[key], list):
                data = data[key]
                break
        else:
            data = [data]

    if not isinstance(data, list):
        return []

    normalized = []
    for item in data:
        if isinstance(item, dict):
            normalized.append(item)
        elif isinstance(item, str) and len(item.strip()) > 1:
            normalized.append({"name": item.strip(), "role": "Supporting", "summary": "Character mentioned in passage."})
            
    return normalized

# ------------------------------------------
# Target Name Node Matcher
# ------------------------------------------
def resolve_to_registry_name(raw_name: str, registry: dict) -> str:
    """Matches raw LLM target names to exact node names in character_registry."""
    if not raw_name or not registry:
        return ""
    
    raw_clean = raw_name.strip()
    raw_lower = raw_clean.lower()
    
    if raw_lower in registry:
        return registry[raw_lower]["name"]
        
    for key, data in registry.items():
        reg_name = data["name"]
        reg_lower = key
        if reg_lower in raw_lower or raw_lower in reg_lower:
            return reg_name
            
    return ""

# ------------------------------------------
# Fast Character & Graph Link Extraction Engine
# ------------------------------------------
def scan_full_book_characters(progress_callback=None) -> str:
    global full_book_chunks, character_registry, book_relationships, full_book_text
    
    if not full_book_chunks:
        return "⚠️ No book indexed yet. Please upload a PDF first."
    
    character_registry = {}
    book_relationships = []

    PRONOUNS_AND_NOISE = {
        "HE", "SHE", "IT", "THEY", "YOU", "WE", "I", "ME", "HIM", "HER", "US", "THEM",
        "He", "She", "It", "They", "You", "We", "I", "Me", "Him", "Her", "Us", "Them",
        "he", "she", "it", "they", "you", "we", "i", "me", "him", "her", "us", "them",
        "HIS", "HERS", "ITS", "OUR", "OURS", "YOUR", "YOURS", "THEIR", "THEIRS", "MY", "MINE",
        "His", "Hers", "Its", "Our", "Ours", "Your", "Yours", "Their", "Theirs", "My", "Mine",
        "Himself", "Herself", "Itself", "Themselves", "Ourselves", "Yourself", "Yourselves", "Myself",
        "This", "That", "These", "Those", "Who", "Whom", "Whose", "Which", "What",
        "Someone", "Somebody", "Something", "Anyone", "Anybody", "Anything",
        "Everyone", "Everybody", "Everything", "No one", "Nobody", "Nothing",
        "The", "And", "Then", "When", "Chapter", "Section", "Where", "With", 
        "After", "Before", "There", "Here", "How", "From", "Into", "About", 
        "Could", "Would", "Should", "Page", "Book", "Part", "But", "Have", 
        "Been", "Were", "Will", "Some", "More", "Most", "Only", "Over", "Such", 
        "Than", "THEN", "THAT", "BUT", "ALSO", "WITH", "FROM", "YES", "NO"
    }
    
    total_chunks = len(full_book_chunks)
    step = max(1, total_chunks // 6)
    sample_indices = list(range(0, total_chunks, step))[:6]
    
    raw_extracted_links = []

    for idx_num, chunk_i in enumerate(sample_indices):
        if progress_callback:
            try:
                progress_callback((idx_num + 1) / len(sample_indices), desc="Extracting character descriptions & relationships...")
            except Exception:
                pass
            
        section_text = full_book_chunks[chunk_i]
        
        prompt = f"""[INST] Extract main proper character names, a brief description of who they are, and their relationships from this excerpt.
Do NOT use pronouns (he, she, they). State clear relationship labels (Lover, Friend, Rival, Family, Enemy, Ally).
Return JSON array:
[
  {{"name": "Proper Name", "summary": "Detailed 1-sentence description of role/actions in text", "related_to": "Other Character Name", "relationship": "Lover / Friend / Rival / Enemy / Family / Ally"}}
]

Excerpt:
{section_text[:1200]} [/INST]"""

        raw_output = invoke_llm(prompt)
        parsed_characters = parse_llm_list(raw_output)
        
        for item in parsed_characters:
            if not isinstance(item, dict):
                continue
                
            name = str(item.get("name", "")).strip()
            if not name or len(name) < 2 or name in PRONOUNS_AND_NOISE or name.lower() in [p.lower() for p in PRONOUNS_AND_NOISE]:
                continue
            
            key = name.lower()
            summary = str(item.get("summary", "")).strip()
            if not summary or summary.lower() in ["character appearing in text.", "none", "unknown"]:
                summary = "Character active in key scenes of the story."
            
            if key not in character_registry:
                character_registry[key] = {
                    "name": name,
                    "role": "Supporting",
                    "traits": ["Character"],
                    "summary": summary,
                    "mentions": 0
                }
            elif len(summary) > len(character_registry[key]["summary"]):
                character_registry[key]["summary"] = summary
            
            related_to = str(item.get("related_to", "")).strip()
            rel_label = str(item.get("relationship", "Connected")).strip()
            
            if related_to and len(related_to) > 2 and related_to not in PRONOUNS_AND_NOISE:
                raw_extracted_links.append((name, related_to, rel_label))

    # Regex Candidate Extraction
    mid_sentence_text = re.sub(r'[\.\?\!\"]\s+[A-Z][a-z]+', '', full_book_text)
    potential_names = re.findall(r'\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)?\b', mid_sentence_text)
    
    counts = {}
    for name in potential_names:
        clean_name = name.strip()
        if clean_name not in PRONOUNS_AND_NOISE and clean_name.lower() not in [p.lower() for p in PRONOUNS_AND_NOISE] and len(clean_name) > 2:
            counts[clean_name] = counts.get(clean_name, 0) + 1
    
    top_entities = sorted(counts.items(), key=lambda x: x[1], reverse=True)[:10]
    
    for name, cnt in top_entities:
        key = name.lower()
        if key not in character_registry:
            match = re.search(r'([^.!?]*?\b' + re.escape(name) + r'\b[^.!?]*[\.!?])', full_book_text)
            snippet = match.group(0).strip().replace('\n', ' ') if match else ""
            
            if snippet and len(snippet) > 15:
                context_summary = f'First mentioned in text: "{snippet}"'
            else:
                context_summary = f"Prominent figure appearing throughout the narrative."
                
            character_registry[key] = {
                "name": name,
                "role": "Supporting",
                "traits": ["Frequent Mention"],
                "summary": context_summary,
                "mentions": cnt
            }

    # Calculate exact total mentions for every character across full book
    for key, c_data in character_registry.items():
        pattern = re.compile(r'\b' + re.escape(c_data["name"]) + r'\b', re.IGNORECASE)
        c_data["mentions"] = len(pattern.findall(full_book_text))

    # Dynamic Role Assignment
    sorted_chars = sorted(character_registry.values(), key=lambda x: x["mentions"], reverse=True)
    total_count = len(sorted_chars)
    
    for idx, c_data in enumerate(sorted_chars):
        if idx < min(3, max(1, int(total_count * 0.15))):
            c_data["role"] = "Main Character / Protagonist"
        elif idx < min(10, max(3, int(total_count * 0.45))):
            c_data["role"] = "Supporting Character"
        else:
            c_data["role"] = "Minor Character"

    # Graph Link Normalization
    for src_raw, tgt_raw, rel_label in raw_extracted_links:
        src_node = resolve_to_registry_name(src_raw, character_registry)
        tgt_node = resolve_to_registry_name(tgt_raw, character_registry)
        
        if src_node and tgt_node and src_node.lower() != tgt_node.lower():
            clean_rel = rel_label if rel_label and rel_label.lower() not in ["none", "unknown", ""] else "Connected"
            
            exists = any(
                (r["source"].lower() == src_node.lower() and r["target"].lower() == tgt_node.lower()) or
                (r["source"].lower() == tgt_node.lower() and r["target"].lower() == src_node.lower())
                for r in book_relationships
            )
            if not exists:
                book_relationships.append({
                    "source": src_node,
                    "target": tgt_node,
                    "relationship": clean_rel
                })

    # Co-occurrence Fallback Link Generation
    reg_names = [c["name"] for c in character_registry.values()]
    if len(book_relationships) < 3 and len(reg_names) > 1:
        paragraphs = full_book_text.split("\n\n")
        co_counts = {}
        
        for para in paragraphs:
            para_names = [n for n in reg_names if re.search(r'\b' + re.escape(n) + r'\b', para, re.IGNORECASE)]
            if len(para_names) >= 2:
                for i in range(len(para_names)):
                    for j in range(i + 1, len(para_names)):
                        pair = tuple(sorted([para_names[i], para_names[j]]))
                        co_counts[pair] = co_counts.get(pair, 0) + 1
                        
        sorted_pairs = sorted(co_counts.items(), key=lambda x: x[1], reverse=True)[:12]
        for (src, tgt), cnt in sorted_pairs:
            exists = any(
                (r["source"].lower() == src.lower() and r["target"].lower() == tgt.lower()) or
                (r["source"].lower() == tgt.lower() and r["target"].lower() == src.lower())
                for r in book_relationships
            )
            if not exists:
                book_relationships.append({
                    "source": src,
                    "target": tgt,
                    "relationship": "Ally / Associate"
                })

    summary_cards = []
    for c_data in sorted_chars:
        summary_cards.append(
            f"👤 **{c_data['name']}** — *{c_data['role']}*\n"
            f"• Mentions: {c_data['mentions']} times\n"
            f"• Summary: {c_data['summary']}\n"
        )

    output_text = f"### 📊 Analysis Complete ({len(character_registry)} Characters & {len(book_relationships)} Graph Connections Identified)\n\n"
    output_text += "\n---\n".join(summary_cards[:15])
    return output_text

# ------------------------------------------
# Plain-Language Paragraph Explainer
# ------------------------------------------
def explain_paragraph(text: str) -> str:
    if not text or not text.strip():
        return "⚠️ Please paste a paragraph to explain."
        
    prompt = f"""[INST] Re-explain the following literary excerpt in clear, simple language.
Break down any subtext or complex phrasing directly and clearly.

Excerpt:
{text} [/INST]"""

    return invoke_llm(prompt)

In [11]:
# ==========================================
# INTERACTIVE RENDERERS & ANALYTICS
# ==========================================
import html
import json
import matplotlib.pyplot as plt
from typing import Dict, List, Any, Tuple

# ------------------------------------------
# Character Cards UI Renderer
# ------------------------------------------
def render_character_cards(registry: Dict[str, Any]) -> str:
    if not registry:
        return "<p style='color:#94a3b8;'>⚠️ No characters analyzed yet — click 'Analyze Full Book Characters'.</p>"

    sorted_chars = sorted(registry.values(), key=lambda x: x.get("mentions", 0), reverse=True)

    cards_html = ""
    for i, c in enumerate(sorted_chars):
        color = CARD_PALETTE[i % len(CARD_PALETTE)]
        escaped_name = html.escape(str(c.get('name', 'Unknown')))
        escaped_role = html.escape(str(c.get('role', 'Character')))
        escaped_summary = html.escape(str(c.get('summary', 'No summary available.')))
        mentions = c.get('mentions', 0)

        cards_html += f"""
        <div onclick="openCharModal('{escaped_name}', '{escaped_role}', '{mentions}', '{escaped_summary}')" 
             style="background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%);
                    border: 1px solid #334155; border-radius: 14px; padding: 1.1rem;
                    margin-bottom: 0.8rem; box-shadow: 0 4px 12px rgba(0,0,0,0.3); cursor: pointer;
                    transition: transform 0.2s ease, border-color 0.2s ease;">
            <div style="display:flex; justify-content:space-between; align-items:center;">
                <h3 style="margin:0; color:{color}; font-size:1.1rem; font-weight:700;">{escaped_name}</h3>
                <span style="background:rgba(99,102,241,0.15); color:#818cf8; font-size:0.75rem; font-weight:600; padding:3px 8px; border-radius:12px;">{mentions} mentions</span>
            </div>
            <div style="color:#94a3b8; font-size:0.8rem; margin:0.3rem 0 0.5rem 0; font-weight:500;">{escaped_role}</div>
            <p style="color:#cbd5e1; font-size:0.88rem; line-height:1.4; margin:0;">{escaped_summary}</p>
        </div>"""

    modal_script = """
    <div id="charModal" style="display:none; position:fixed; top:0; left:0; width:100%; height:100%; background:rgba(0,0,0,0.8); backdrop-filter:blur(4px); z-index:9999; justify-content:center; align-items:center;">
        <div style="background:#1e293b; border:1px solid #475569; border-radius:16px; padding:2rem; max-width:500px; width:90%; color:#f8fafc; position:relative; box-shadow: 0 20px 25px -5px rgba(0,0,0,0.5);">
            <button onclick="closeCharModal()" style="position:absolute; top:12px; right:16px; background:none; border:none; color:#94a3b8; font-size:1.5rem; cursor:pointer;">&times;</button>
            <h2 id="modalName" style="margin-top:0; color:#818cf8;"></h2>
            <p><strong>Role:</strong> <span id="modalRole" style="color:#cbd5e1;"></span></p>
            <p><strong>Total Mentions in Book:</strong> <span id="modalMentions" style="color:#38bdf8;"></span></p>
            <hr style="border-color:#334155; margin:1rem 0;"/>
            <p><strong>Overview:</strong></p>
            <p id="modalSummary" style="color:#e2e8f0; line-height:1.5;"></p>
        </div>
    </div>
    <script>
    function openCharModal(name, role, mentions, summary) {
        document.getElementById('modalName').innerText = name;
        document.getElementById('modalRole').innerText = role;
        document.getElementById('modalMentions').innerText = mentions;
        document.getElementById('modalSummary').innerText = summary;
        document.getElementById('charModal').style.display = 'flex';
    }
    function closeCharModal() {
        document.getElementById('charModal').style.display = 'none';
    }
    </script>
    """

    return f"<div style='max-height:550px; overflow-y:auto; padding-right:6px;'>{modal_script}{cards_html}</div>"

# ------------------------------------------
# Force-Directed Vis.js Graph Renderer
# ------------------------------------------
def render_character_graph(registry: Dict[str, Any], relationships: List[Dict[str, str]]) -> str:
    if not registry:
        return "<div style='color:#94a3b8; padding:1rem;'>⚠️ No character network available yet.</div>"

    nodes, edges, added = [], [], {}
    for i, (key, c) in enumerate(registry.items()):
        color = CARD_PALETTE[i % len(CARD_PALETTE)]
        size = 20 + min(c.get("mentions", 0), 35)
        nodes.append({
            "id": c["name"], 
            "label": c["name"],
            "color": {"background": color, "border": "#ffffff", "highlight": {"background": "#6366f1", "border": "#ffffff"}},
            "shape": "dot", 
            "size": size,
            "font": {"color": "#f8fafc", "size": 13, "face": "Inter"}
        })
        added[key] = True

    for r in relationships:
        if not isinstance(r, dict):
            continue
        src = str(r.get("source", ""))
        tgt = str(r.get("target", ""))
        rel = str(r.get("relationship", "Connected"))
        if src.lower() in added and tgt.lower() in added:
            edges.append({
                "from": src, "to": tgt,
                "label": rel,
                "color": {"color": "#475569", "highlight": "#818cf8"},
                "font": {"color": "#94a3b8", "size": 10, "align": "middle"},
                "width": 2
            })

    raw_html = f"""<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8">
<script src="https://cdnjs.cloudflare.com/ajax/libs/vis-network/9.1.2/standalone/umd/vis-network.min.js"></script>
<style>
  html, body {{ margin:0; padding:0; width:100%; height:100%; background:#0f172a; font-family:sans-serif; }}
  #mynetwork {{ width:100%; height:100%; background:#0f172a; }}
</style>
</head>
<body>
<div id="mynetwork"></div>
<script type="text/javascript">
  var container = document.getElementById('mynetwork');
  var data = {{
      nodes: new vis.DataSet({json.dumps(nodes)}),
      edges: new vis.DataSet({json.dumps(edges)})
  }};
  var options = {{
      nodes: {{ borderWidth: 2, shadow: true }},
      edges: {{ smooth: {{ type: 'continuous' }} }},
      physics: {{
          solver: 'forceAtlas2Based',
          forceAtlas2Based: {{ gravitationalConstant: -60, centralGravity: 0.01, springLength: 100, springConstant: 0.08 }}
      }}
  }};
  var network = new vis.Network(container, data, options);
</script>
</body>
</html>"""

    return f'<iframe srcdoc="{html.escape(raw_html)}" style="width:100%; height:520px; border:1px solid #1e293b; border-radius:14px; background:#0f172a;"></iframe>'

# ------------------------------------------
# Catch-Up, Summary, Quiz & Recommendation
# ------------------------------------------
def generate_catchup_summary(stopped_section: int) -> str:
    global full_book_chunks
    if not full_book_chunks:
        return "⚠️ Please upload and index a book first."
    
    total_chunks = len(full_book_chunks)
    target_chunks_count = min(int(stopped_section * 5), total_chunks)
    
    if target_chunks_count == 0:
        return "⚠️ Selected section is too low."
    
    recap_text = "\n\n".join(full_book_chunks[:target_chunks_count])
    
    prompt = f"""[INST] Generate a spoiler-free "Previously in this book..." summary up to Section {stopped_section}.

Context:
{recap_text[:3500]} [/INST]"""

    return invoke_llm(prompt)

def summarize_text(text: str) -> str:
    if not text or not text.strip():
        return "⚠️ Please paste text to summarize."
    
    prompt = f"""[INST] Provide a concise bulleted summary (4-6 points) from this excerpt:

{text[:3000]} [/INST]"""

    return invoke_llm(prompt)

def generate_pydantic_quiz(text_sample: str) -> str:
    global full_book_text
    
    if not text_sample or not text_sample.strip():
        text_sample = full_book_text[:3000] if full_book_text else ""
    if not text_sample:
        return "⚠️ No text uploaded or provided for quiz generation."

    prompt = f"""[INST] Generate 3 multiple-choice comprehension questions strictly based on the text.
Format output strictly as JSON list:
[
  {{
    "question": "Question text...",
    "options": ["A) Opt 1", "B) Opt 2", "C) Opt 3", "D) Opt 4"],
    "answer": "A) Opt 1"
  }}
]

Text:
{text_sample[:3000]} [/INST]"""

    raw = invoke_llm(prompt)
    data = parse_llm_list(raw)
    
    if not data:
        return f"Raw Output:\n\n{raw}"

    formatted = "### 🎯 Interactive Comprehension Quiz\n\n"
    for i, q in enumerate(data, 1):
        if not isinstance(q, dict):
            continue
        formatted += f"**Q{i}: {q.get('question', 'Question')}**\n"
        for opt in q.get('options', []):
            formatted += f"- {opt}\n"
        formatted += f"\n<details><summary><b>View Correct Answer</b></summary><i>{q.get('answer', 'N/A')}</i></details>\n\n---\n"
    return formatted

def render_recommendation_cards(recs: list) -> str:
    if not recs or not isinstance(recs, list):
        return "<p style='color:#94a3b8;'>⚠️ No recommendations generated yet.</p>"

    cards = ""
    for i, r in enumerate(recs):
        if not isinstance(r, dict):
            continue
        color = CARD_PALETTE[i % len(CARD_PALETTE)]
        title = html.escape(str(r.get('title', 'Unknown Title')))
        author = html.escape(str(r.get('author', 'Unknown Author')))
        why_fits = html.escape(str(r.get('why_it_fits', '')))
        premise = html.escape(str(r.get('premise', '')))

        cards += f"""
        <div style="background:linear-gradient(135deg, #1e293b 0%, #0f172a 100%); border-left:4px solid {color}; border-radius:12px; padding:1.1rem; margin-bottom:0.9rem;">
            <h3 style="margin:0; color:{color};">{title}</h3>
            <div style="color:#94a3b8; font-size:0.85rem; margin-bottom:0.5rem;">by {author}</div>
            <p style="color:#e2e8f0; font-size:0.9rem; margin:0 0 0.4rem 0;"><b>Why it fits:</b> {why_fits}</p>
            <p style="color:#cbd5e1; font-size:0.85rem; margin:0;">{premise}</p>
        </div>"""
    return f"<div>{cards}</div>"

def recommend_books(emotions: str, pacing: str, ending: str, protagonist: str, length: str, genre: str) -> str:
    prompt = f"""[INST] Recommend 3 real books matching:
- Vibe: {emotions or 'Any'}
- Pacing: {pacing}
- Ending: {ending}
- Protagonist: {protagonist or 'Any'}
- Length: {length}
- Genre: {genre or 'General Fiction'}

Format output strictly as JSON array:
[
  {{
    "title": "Title",
    "author": "Author",
    "why_it_fits": "Reason",
    "premise": "Premise"
  }}
] [/INST]"""

    raw_output = invoke_llm(prompt)
    recs = parse_llm_list(raw_output)
    return render_recommendation_cards(recs)

# ------------------------------------------
# Analytics Summary & Matplotlib Plotting
# ------------------------------------------
def compute_analytics() -> Tuple[str, plt.Figure]:
    global full_book_text, full_book_chunks, character_registry

    if not full_book_text:
        fig, ax = plt.subplots(figsize=(6, 3.5))
        fig.patch.set_facecolor('#0f172a')
        ax.set_facecolor('#0f172a')
        ax.text(0.5, 0.5, "No Book Uploaded Yet", color="#94a3b8", ha='center', va='center', fontsize=12)
        ax.axis('off')
        empty_stats = "<div style='color:#94a3b8; padding:1rem;'>⚠️ Please upload and index a PDF first.</div>"
        return empty_stats, fig

    total_words = len(full_book_text.split())
    est_reading_hours = round(total_words / 225 / 60, 1)
    num_chunks = len(full_book_chunks)

    stats_html = f"""
    <div style="background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%); border:1px solid #334155; border-radius:14px; padding:1.2rem; color:#f8fafc; margin-bottom:1rem; box-shadow:0 4px 12px rgba(0,0,0,0.3);">
        <h4 style="margin:0 0 0.8rem 0; color:#818cf8; font-size:1.05rem; border-bottom:1px solid #334155; padding-bottom:0.4rem; font-weight:700;">
            📊 Book Intelligence Summary
        </h4>
        <div style="display:grid; grid-template-columns:1fr 1fr; gap:0.8rem; font-size:0.85rem;">
            <div><span style="color:#94a3b8;">Total Words:</span><br/><b style="color:#38bdf8; font-size:1.1rem;">{total_words:,}</b></div>
            <div><span style="color:#94a3b8;">Est. Read Time:</span><br/><b style="color:#34d399; font-size:1.1rem;">{est_reading_hours} hrs</b></div>
            <div><span style="color:#94a3b8;">Total Chunks:</span><br/><b style="color:#f472b6; font-size:1.1rem;">{num_chunks}</b></div>
            <div><span style="color:#94a3b8;">Characters Found:</span><br/><b style="color:#fbbf24; font-size:1.1rem;">{len(character_registry)}</b></div>
        </div>
    </div>
    """

    fig, ax = plt.subplots(figsize=(6, 3.5))
    fig.patch.set_facecolor('#0f172a')
    ax.set_facecolor('#0f172a')

    if character_registry:
        sorted_chars = sorted(character_registry.values(), key=lambda x: x.get("mentions", 0), reverse=True)[:8]
        names = [c["name"] for c in sorted_chars][::-1]
        mentions = [c.get("mentions", 0) for c in sorted_chars][::-1]

        ax.barh(names, mentions, color='#818cf8', edgecolor='#c7d2fe', height=0.6)
        ax.set_title("Top Characters by Mentions", color="#f8fafc", fontsize=11, fontweight='bold', pad=10)
        ax.set_xlabel("Mention Frequency", color="#cbd5e1", fontsize=9)
        ax.tick_params(colors="#cbd5e1", labelsize=8)
        ax.grid(True, axis='x', color='#1e293b', linestyle='--', alpha=0.7)
        for spine in ax.spines.values():
            spine.set_color('#334155')
    else:
        ax.text(0.5, 0.5, "Run 'Analyze Full Book'\nto populate character chart", 
                color="#64748b", ha='center', va='center', fontsize=10)
        ax.axis('off')

    plt.tight_layout()
    return stats_html, fig

In [12]:
# ==========================================
# GRADIO DASHBOARD & ANIMATED UI
# ==========================================
import os
import gradio as gr

custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap');

* {
    font-family: 'Plus Jakarta Sans', sans-serif !important;
}

body, .gradio-container {
    background-color: #030712 !important;
    color: #f3f4f6 !important;
}

/* Keyframe Animations */
@keyframes gradientShift {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

@keyframes pulseGlow {
    0% { box-shadow: 0 0 10px rgba(99, 102, 241, 0.4); }
    50% { box-shadow: 0 0 22px rgba(129, 140, 248, 0.7); }
    100% { box-shadow: 0 0 10px rgba(99, 102, 241, 0.4); }
}

@keyframes floatSlow {
    0% { transform: translateY(0px); }
    50% { transform: translateY(-4px); }
    100% { transform: translateY(0px); }
}

/* Animated Header */
.gradio-header {
    background: linear-gradient(-45deg, #0f172a, #1e1b4b, #311042, #020617);
    background-size: 300% 300%;
    animation: gradientShift 12s ease infinite;
    border: 1px solid rgba(129, 140, 248, 0.3);
    padding: 2rem;
    border-radius: 20px;
    margin-bottom: 1.2rem;
    box-shadow: 0 10px 30px rgba(0, 0, 0, 0.5);
}

/* Animated Primary Buttons */
.main-btn {
    background: linear-gradient(135deg, #6366f1 0%, #4f46e5 50%, #4338ca 100%) !important;
    background-size: 200% 200% !important;
    color: #ffffff !important;
    font-weight: 700 !important;
    border: 1px solid rgba(255, 255, 255, 0.2) !important;
    border-radius: 12px !important;
    padding: 12px 24px !important;
    transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1) !important;
    animation: pulseGlow 4s infinite !important;
    cursor: pointer !important;
}

.main-btn:hover {
    transform: translateY(-3px) scale(1.02) !important;
    box-shadow: 0 10px 25px rgba(99, 102, 241, 0.6) !important;
    background-position: right center !important;
}

.main-btn:active {
    transform: translateY(1px) scale(0.98) !important;
}

/* Animated Secondary Buttons */
.secondary-btn {
    background: linear-gradient(135deg, #1e293b 0%, #0f172a 100%) !important;
    color: #e2e8f0 !important;
    border: 1px solid #334155 !important;
    border-radius: 12px !important;
    font-weight: 600 !important;
    transition: all 0.25s ease !important;
}

.secondary-btn:hover {
    background: #334155 !important;
    border-color: #818cf8 !important;
    color: #ffffff !important;
    transform: translateY(-2px) !important;
}

/* Glassmorphism Containers & Cards */
.sidebar-panel {
    background: linear-gradient(180deg, rgba(15, 23, 42, 0.9) 0%, rgba(3, 7, 18, 0.95) 100%) !important;
    border: 1px solid rgba(51, 65, 85, 0.6) !important;
    border-radius: 18px !important;
    padding: 1.4rem !important;
    backdrop-filter: blur(12px) !important;
}

.tab-nav button {
    font-weight: 700 !important;
    font-size: 0.95rem !important;
    color: #94a3b8 !important;
    transition: all 0.2s ease !important;
}

.tab-nav button.selected {
    color: #818cf8 !important;
    border-bottom: 3px solid #6366f1 !important;
    text-shadow: 0 0 10px rgba(99, 102, 241, 0.5) !important;
}

/* Inputs & Textareas Hover/Focus Glow */
textarea, input[type="text"] {
    background-color: #0f172a !important;
    border: 1px solid #334155 !important;
    border-radius: 10px !important;
    color: #f8fafc !important;
    transition: border-color 0.25s ease, box-shadow 0.25s ease !important;
}

textarea:focus, input[type="text"]:focus {
    border-color: #6366f1 !important;
    box-shadow: 0 0 12px rgba(99, 102, 241, 0.3) !important;
}
"""

with gr.Blocks(css=custom_css, title="Novellm") as demo:
    
    # Animated Header Banner
    gr.HTML("""
    <div class="gradio-header">
        <div style="display:flex; align-items:center; justify-content:space-between;">
            <div style="display:flex; align-items:center; gap:16px;">
                <div style="animation: floatSlow 3s ease-in-out infinite; font-size:2.6rem;">✨</div>
                <div>
                    <h1 style="margin:0; color:#e0e7ff; font-size:2.1rem; font-weight:800; letter-spacing:-0.5px;">
                        Novellm
                    </h1>
                    <p style="margin:0.3rem 0 0 0; color:#a5b4fc; font-size:0.95rem; font-weight:500;">
                        Interactive RAG Engine • Dynamic Character Graphs • Smart Explanations
                    </p>
                </div>
            </div>
            <div style="display:flex; align-items:center; gap:8px; background:rgba(15, 23, 42, 0.6); padding:8px 16px; border-radius:20px; border:1px solid rgba(129, 140, 248, 0.3);">
                <span style="height:10px; width:10px; background-color:#34d399; border-radius:50%; display:inline-block; box-shadow: 0 0 8px #34d399;"></span>
                <span style="color:#cbd5e1; font-size:0.82rem; font-weight:600;">System Ready</span>
            </div>
        </div>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=3):
            with gr.Tabs():
                
                # TAB 1: Reader Hub
                with gr.TabItem("📖 Reader Hub"):
                    gr.Markdown("### 1. Ingest & Index Book")
                    with gr.Row():
                        pdf_input = gr.File(label="Upload Book (PDF)", file_types=[".pdf"])
                        index_btn = gr.Button("⚡ Index Book", variant="primary", elem_classes=["main-btn"])
                    index_output = gr.Textbox(label="Indexing Status", interactive=False, lines=3)
                    
                    gr.Markdown("---")
                    gr.Markdown("### 2. Ask Questions (RAG)")
                    qa_input = gr.Textbox(label="Question about the book", placeholder="e.g. What key decision does the main character make in Chapter 2?")
                    qa_btn = gr.Button("🔍 Answer Question", elem_classes=["main-btn"])
                    qa_output = gr.Textbox(label="Answer", interactive=False, lines=4)
                    
                    gr.Markdown("---")
                    gr.Markdown("### 3. Spoiler-Free Catch-Up Tracker")
                    with gr.Row():
                        stopped_slider = gr.Slider(minimum=1, maximum=50, step=1, value=5, label="I stopped at Section / Chapter:")
                        catchup_btn = gr.Button("📜 Generate Catch-Up Summary", elem_classes=["secondary-btn"])
                    catchup_output = gr.Textbox(label="Previously in this book...", interactive=False, lines=6)

                # TAB 2: Character Network
                with gr.TabItem("👥 Character Network"):
                    gr.Markdown("### Full-Book Character & Network Analysis")
                    gr.Markdown("Extracts character profiles and builds an interactive relationship network.")
                    scan_char_btn = gr.Button("🔍 Analyze Full Book Characters", variant="primary", elem_classes=["main-btn"])
                    
                    with gr.Accordion("📋 Extraction Status & Diagnostics", open=True):
                        char_status = gr.Textbox(label="Status Output", interactive=False, lines=3)
                    
                    with gr.Row():
                        with gr.Column(scale=1):
                            gr.Markdown("#### Character Cards")
                            char_cards_html = gr.HTML(value="<p style='color:#64748b;'>Upload a book and click Analyze to view character profiles.</p>")
                        with gr.Column(scale=1):
                            gr.Markdown("#### Interaction Graph")
                            char_graph_html = gr.HTML(value="<p style='color:#64748b;'>Character network will appear here.</p>")

                # TAB 3: Summaries & Quizzes
                with gr.TabItem("📝 Summaries & Quizzes"):
                    gr.Markdown("### Excerpt Summarizer & Quiz Generator")
                    summary_text_input = gr.Textbox(
                        label="Paste Text Excerpt (Optional - leave empty to use book text)",
                        placeholder="Paste a passage to summarize or generate a quiz...",
                        lines=5
                    )
                    with gr.Row():
                        summarize_btn = gr.Button("📋 Summarize Excerpt", elem_classes=["secondary-btn"])
                        quiz_btn = gr.Button("🎯 Generate Quiz", elem_classes=["main-btn"])
                    
                    summary_output = gr.Textbox(label="Bullet Summary", interactive=False, lines=5)
                    quiz_output = gr.Markdown(label="Comprehension Quiz")

                # TAB 4: Plain-Language Explainer
                with gr.TabItem("💡 Explain This"):
                    gr.Markdown("### Simplify Dense Passages")
                    explain_input = gr.Textbox(label="Complex Text Excerpt", placeholder="Paste confusing paragraph here...", lines=4)
                    explain_btn = gr.Button("💡 Simplify Paragraph", elem_classes=["main-btn"])
                    explain_output = gr.Textbox(label="Simplified Explanation", interactive=False, lines=5)

                # TAB 5: Book Recommender
                with gr.TabItem("📚 Book Recommender"):
                    gr.Markdown("### Find Your Next Favorite Read")
                    with gr.Row():
                        rec_emotions = gr.Textbox(label="Desired Vibe/Emotions", placeholder="e.g. melancholic, dark, cozy, hopeful")
                        rec_pacing = gr.Dropdown(choices=["Fast-Paced", "Medium-Paced", "Slow Burn"], value="Medium-Paced", label="Pacing")
                        rec_ending = gr.Dropdown(choices=["Happy/Satisfying", "Bittersweet", "Twist/Shocking", "Open-Ended"], value="Happy/Satisfying", label="Ending Type")
                    with gr.Row():
                        rec_protagonist = gr.Textbox(label="Protagonist Archetype", placeholder="e.g. reluctant hero, morally grey, detective")
                        rec_length = gr.Dropdown(choices=["Short (<250 pages)", "Medium (250-450 pages)", "Epic (>450 pages)"], value="Medium (250-450 pages)", label="Length")
                        rec_genre = gr.Textbox(label="Genre or Tropes", placeholder="e.g. Sci-Fi, enemies-to-lovers, mystery")
                    
                    rec_btn = gr.Button("✨ Discover Books", elem_classes=["main-btn"])
                    rec_output = gr.HTML()

        # Sidebar Panel
        with gr.Column(scale=1, elem_classes=["sidebar-panel"]):
            gr.Markdown("### 📊 Analytics Sidebar")
            refresh_analytics_btn = gr.Button("🔄 Refresh Stats", elem_classes=["secondary-btn"])
            sidebar_stats_html = gr.HTML(value="<div style='color:#94a3b8;'>No book loaded yet.</div>")
            sidebar_chart = gr.Plot(label="Mention Frequency")

    # Callbacks & Event Handlers
    def handle_indexing(file):
        status = process_pdf(file)
        stats_h, chart = compute_analytics()
        return status, stats_h, chart

    def handle_char_analysis(progress=gr.Progress()):
        status = scan_full_book_characters(progress_callback=progress)
        cards = render_character_cards(character_registry)
        graph = render_character_graph(character_registry, book_relationships)
        stats_h, chart = compute_analytics()
        return status, cards, graph, stats_h, chart

    index_btn.click(
        fn=handle_indexing,
        inputs=[pdf_input],
        outputs=[index_output, sidebar_stats_html, sidebar_chart]
    )

    qa_btn.click(
        fn=answer_rag_question,
        inputs=[qa_input],
        outputs=[qa_output]
    )

    catchup_btn.click(
        fn=generate_catchup_summary,
        inputs=[stopped_slider],
        outputs=[catchup_output]
    )

    scan_char_btn.click(
        fn=handle_char_analysis,
        inputs=[],
        outputs=[char_status, char_cards_html, char_graph_html, sidebar_stats_html, sidebar_chart]
    )

    summarize_btn.click(
        fn=summarize_text,
        inputs=[summary_text_input],
        outputs=[summary_output]
    )

    quiz_btn.click(
        fn=generate_pydantic_quiz,
        inputs=[summary_text_input],
        outputs=[quiz_output]
    )

    explain_btn.click(
        fn=explain_paragraph,
        inputs=[explain_input],
        outputs=[explain_output]
    )

    rec_btn.click(
        fn=recommend_books,
        inputs=[rec_emotions, rec_pacing, rec_ending, rec_protagonist, rec_length, rec_genre],
        outputs=[rec_output]
    )

    refresh_analytics_btn.click(
        fn=compute_analytics,
        inputs=[],
        outputs=[sidebar_stats_html, sidebar_chart]
    )

print("🚀 Launching Animated Gradio Web Dashboard...")
demo.queue().launch(share=True, debug=True)

/tmp/ipykernel_58/3248341214.py:128: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=custom_css, title="Novellm") as demo:


🚀 Launching Animated Gradio Web Dashboard...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://eb130ad065165aeb9c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=700) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=700) and `max_

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://eb130ad065165aeb9c.gradio.live
